## CCA-F Lab 02: Herramientas en Paralelo y Modos de tool_choice (Días 3 y 4)

### Objetivos de Aprendizaje para el Examen:
1. **Parallel Tool Calls:** Claude puede solicitar múltiples ejecuciones de herramientas en un solo turno (ej: recopilar telemetría de varios servidores a la vez). Esto reduce los "round-trips" HTTP, bajando costos y latencia.
2. **Regla de Correspondencia Exacta (1:1):** Si Claude genera un mensaje con 3 bloques de `tool_use`, el desarrollador **debe** responder en el siguiente turno con un mensaje que contenga exactamente 3 bloques de `tool_used` mapeados mediante sus `tool_use_id`. Si falta uno o mandas de más, la API fallará con un error 400.
3. **Modos de `tool_choice`:**
   * `auto`: (Por defecto) El modelo decide libremente si usa herramientas o responde con texto plano.
   * `any`: Obliga a Claude a llamar a *al menos una* herramienta del catálogo, pero él elige cuál.
   * `forced` (`{"type": "tool", "name": "..."}`): Obliga a Claude a llamar a una herramienta *específica*. Ideal para transformar texto no estructurado en JSON puro sin prosa.

In [1]:
import os
import json
from anthropic import Anthropic
from dotenv import load_dotenv

# Cargar variables de entorno desde la raíz del proyecto
load_dotenv(dotenv_path="../.env")

client = Anthropic()
print("✅ Cliente de Anthropic configurado.")

✅ Cliente de Anthropic configurado.


In [2]:
# Backend: Funciones simuladas
def get_customer_tier(customer_id: str) -> dict:
    """Simula consulta de base de datos de clientes."""
    print(f"⚙️ [Backend Exec] Ejecutando DB Query para cliente: {customer_id}")
    database = {
        "CUST-100": {"company": "Acme Corp", "tier": "ENTERPRISE", "sla_hours": 2},
        "CUST-200": {"company": "Stark Industries", "tier": "GOLD", "sla_hours": 4}
    }
    return database.get(customer_id, {"error": "Customer profile not found"})

def get_network_latency(region: str) -> dict:
    """Simula monitoreo de infraestructura de red."""
    print(f"⚙️ [Backend Exec] Consultando telemetría de red en: {region}")
    regions = {
        "us-east-1": {"latency_ms": 15, "packet_loss_pct": 0.0},
        "eu-west-1": {"latency_ms": 340, "packet_loss_pct": 4.2},  # Anomalía degradada
    }
    return regions.get(region.lower(), {"error": "Region telemetry unavailable"})

def save_analytics_report(customer_id: str, priority_level: str, technical_summary: str) -> dict:
    """Herramienta para Extracción Estructurada Forzada (Forced tool_choice)."""
    print(f"💾 [Backend Exec] Guardando reporte final estructurado en Data Lake para {customer_id}...")
    return {"status": "RECORDED", "payload_received": {
        "customer_id": customer_id,
        "priority": priority_level,
        "summary": technical_summary
    }}

# Esquemas de la API de Anthropic
TOOLS_CATALOG = [
    {
        "name": "get_customer_tier",
        "description": "Retrieves the subscription level, company name, and contractual SLA response hours for a customer using their strict ID format (e.g., CUST-100).",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The unique enterprise identifier."}
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "get_network_latency",
        "description": "Checks network performance metrics including packet loss and latency for a specific cloud region provider code (e.g., 'us-east-1', 'eu-west-1').",
        "input_schema": {
            "type": "object",
            "properties": {
                "region": {"type": "string", "description": "Cloud deployment region designation."}
            },
            "required": ["region"]
        }
    },
    {
        "name": "save_analytics_report",
        "description": "Saves a definitive, machine-readable structured operational incident analysis directly into the infrastructure data lake.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "priority_level": {"type": "string", "enum": ["CRITICAL", "HIGH", "NORMAL"]},
                "technical_summary": {"type": "string", "description": "Synthesized diagnosis of the structural incident."}
            },
            "required": ["customer_id", "priority_level", "technical_summary"]
        }
    }
]
print("🔧 Catálogo de herramientas registrado.")

🔧 Catálogo de herramientas registrado.


In [5]:
def run_advanced_tool_pipeline(user_prompt: str):
    messages = [{"role": "user", "content": user_prompt}]
    
    # --- TURNO 1: LLAMADAS EN PARALELO ---
    print("\n--- 🛰️ ENVIANDO TURNO 1 A CLAUDE (Modo tool_choice: auto) ---")
    response = client.messages.create(
        model="claude-sonnet-4-6", # Usamos Claude Sonnet 4.6 para esta prueba
        max_tokens=2000,
        system="You are an automated multi-datacenter incident coordinator.",
        tools=TOOLS_CATALOG,
        tool_choice={"type": "auto"},
        messages=messages
    )
    
    # Registrar la respuesta del asistente en el ledger
    messages.append({"role": "assistant", "content": response.content})
    
    # Procesar bloques solicitados por Claude
    tool_use_blocks = [b for b in response.content if b.type == "tool_use"]
    print(f"📊 Claude solicitó {len(tool_use_blocks)} herramientas en este turno único.")
    
    # Crear la lista de resultados para armar el mensaje de respuesta estructurado del usuario
    tool_results_content = []
    
    for block in tool_use_blocks:
        print(f"👉 Solicitud Detectada: {block.name} (ID: {block.id})")
        
        # Ejecutar en el backend local
        if block.name == "get_customer_tier":
            res = get_customer_tier(**block.input)
        elif block.name == "get_network_latency":
            res = get_network_latency(**block.input)
        else:
            res = {"error": "Unknown tool invocation"}
            
        # REGLA DE EXAMEN: Añadir cada resultado individual referenciando su ID correspondiente
        tool_results_content.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": json.dumps(res)
        })
    
    # Adjuntar todas las respuestas en UN solo mensaje con rol 'user' (Cumple contrato 1:1)
    messages.append({
        "role": "user",
        "content": tool_results_content
    })
    print("✅ Todas las respuestas de herramientas consolidadas en el ledger de mensajes.")
    
    # --- TURNO 2: EXTRACCIÓN JSON FINAL CON MODO FORZADO (Forced tool_choice) ---
    print("\n--- 🎯 ENVIANDO TURNO 2 A CLAUDE (Modo tool_choice: forced) ---")
    
    # Aquí cambiamos la configuración de tool_choice para obligar a guardar los datos estructurados
    forced_response = client.messages.create(
        model="claude-sonnet-4-6", # Usamos Claude Sonnet 4.6 para esta prueba
        max_tokens=2000,
        system="Analyze the collected data. You MUST save the final incident analytics structure now.",
        tools=TOOLS_CATALOG,
        tool_choice={"type": "tool", "name": "save_analytics_report"}, # EXAM CORE: Selección forzada
        messages=messages
    )
    
    # Imprimir la salida estructurada garantizada
    print("\n📦 [RESPUESTA DE CLAUDE BAJO CONFIGURACIÓN FORZADA]:")
    for block in forced_response.content:
        if block.type == "text":
            print(f"Texto (No debería haber prosa): {block.text}")
        elif block.type == "tool_use":
            print(f"Nombre de Herramienta Invocada: {block.name}")
            print(f"Argumentos JSON puros generados: {json.dumps(block.input, indent=2)}")

# Ejecutar el experimento de producción
prompt_caso_complejo = (
    "El cliente corporativo con identificador 'CUST-200' está reportando caídas "
    "en sus servicios desplegados en la región 'eu-west-1'. Recopila los datos de ambos "
    "elementos de forma inmediata para analizar el impacto técnico según su contrato."
)

run_advanced_tool_pipeline(prompt_caso_complejo)


--- 🛰️ ENVIANDO TURNO 1 A CLAUDE (Modo tool_choice: auto) ---
📊 Claude solicitó 2 herramientas en este turno único.
👉 Solicitud Detectada: get_customer_tier (ID: toolu_01JfkeFuEnjDKNgwwZX3iv2Z)
⚙️ [Backend Exec] Ejecutando DB Query para cliente: CUST-200
👉 Solicitud Detectada: get_network_latency (ID: toolu_013h3VEduNEvTn2s31nsKUeF)
⚙️ [Backend Exec] Consultando telemetría de red en: eu-west-1
✅ Todas las respuestas de herramientas consolidadas en el ledger de mensajes.

--- 🎯 ENVIANDO TURNO 2 A CLAUDE (Modo tool_choice: forced) ---

📦 [RESPUESTA DE CLAUDE BAJO CONFIGURACIÓN FORZADA]:
Nombre de Herramienta Invocada: save_analytics_report
Argumentos JSON puros generados: {
  "customer_id": "CUST-200",
  "priority_level": "CRITICAL",
  "technical_summary": "Incident Report - Stark Industries (CUST-200) | Tier: GOLD | SLA: 4h response. Region eu-west-1 shows degraded network performance: latency at 340ms (severely above acceptable threshold) and packet loss at 4.2% (critical threshold ex

1. En el Turno 1, debido a que tu prompt le pide investigar dos cosas independientes (CUST-200 y eu-west-1), Claude 3.5 Sonnet inteligentemente generará dos bloques tool_use de forma simultánea en una sola respuesta.

2. Tu código recolectará ambos IDs y enviará un arreglo con dos respuestas tool_result bajo el rol user.

3. En el Turno 2, gracias al parámetro tool_choice: {"type": "tool", "name": "save_analytics_report"}, Claude no te devolverá saludos ni explicaciones conversacionales; te entregará directamente el objeto con el resumen técnico mapeado al esquema estricto de la herramienta.